# 🆚 Raw OpenAI API vs OpenAI Agents SDK

**Same task, two approaches** — so you can see exactly what the SDK buys you.

### The task
Build a mini **"Career Coach"** pipeline:
1. Three agents write different coaching tips (professional / motivational / concise)
2. A picker agent selects the best one
3. A guardrail blocks off-topic questions
4. A tool records when the user asks to save a tip

This mirrors exactly what `1_foundations/3_lab3.ipynb` + `2_openai/2_lab2.ipynb` + `2_openai/3_lab3.ipynb` build — but side-by-side.

In [1]:
import os, json, asyncio
from dotenv import load_dotenv
load_dotenv(override=True)

True

---
## ❌ APPROACH 1 — Raw API (1_foundations style)

No framework. Just `openai.OpenAI` and manual wiring.

In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
MODEL = "openai/gpt-4o-mini"

# ── System prompts ──────────────────────────────────────────────
SYSTEM_PROFESSIONAL = "You are a professional career coach. Give formal, structured career advice."
SYSTEM_MOTIVATIONAL = "You are an energetic motivational career coach. Give uplifting, passionate advice."
SYSTEM_CONCISE      = "You are a busy career coach. Give advice in bullet points, under 60 words."
SYSTEM_PICKER       = (
    "You are given 3 career tips. Pick the best one for a junior developer. "
    "Reply with ONLY the selected tip, no explanation."
)
SYSTEM_GUARDRAIL    = (
    "You decide if a question is career-related. "
    "Reply with JSON: {\"is_career_related\": true} or {\"is_career_related\": false}"
)

# ── 🔥 Problem 1: Every agent is just copy-pasted boilerplate ───
def call_agent(system: str, user: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ]
    )
    return response.choices[0].message.content

# ── 🔥 Problem 2: Tool use is all manual JSON wiring ────────────
save_tool_schema = {
    "type": "function",
    "function": {
        "name": "save_tip",
        "description": "Save a career tip to the user's notebook",
        "parameters": {
            "type": "object",
            "properties": {
                "tip": {"type": "string", "description": "The tip to save"}
            },
            "required": ["tip"]
        }
    }
}

def save_tip(tip: str) -> dict:
    print(f"[TOOL] Saving tip: {tip[:60]}...")
    return {"saved": True}

# ── 🔥 Problem 3: Guardrail = a separate manual LLM call ────────
def is_career_related(question: str) -> bool:
    raw = call_agent(SYSTEM_GUARDRAIL, question)
    # Hope the LLM returned valid JSON... might need stripping...
    cleaned = raw.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(cleaned)["is_career_related"]
    except Exception:
        return True  # fail open — dangerous

# ── 🔥 Problem 4: Parallel calls = manual asyncio threading ─────
# (Can't easily do this synchronously, so we just do it sequentially here)
def raw_pipeline(question: str) -> str:
    # Step 0: Guardrail check
    if not is_career_related(question):
        return "Sorry, I only answer career-related questions."

    # Step 1: Three agents write tips — sequential, slow
    tip1 = call_agent(SYSTEM_PROFESSIONAL, question)
    tip2 = call_agent(SYSTEM_MOTIVATIONAL, question)
    tip3 = call_agent(SYSTEM_CONCISE, question)

    # Step 2: Picker selects best
    combined = f"Tip 1:\n{tip1}\n\nTip 2:\n{tip2}\n\nTip 3:\n{tip3}"
    best = call_agent(SYSTEM_PICKER, combined)

    # Step 3: Tool call — also manual
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "If the user wants to save a tip, use the save_tip tool."},
            {"role": "user",   "content": f"Please save this tip: {best}"}
        ],
        tools=[save_tool_schema]
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            save_tip(**args)  # manual dispatch

    return best

# No tracing, no logging — if something breaks, good luck.
result_raw = raw_pipeline("How should I prepare for my first software engineering job?")
print("=== RAW RESULT ===")
print(result_raw)

[TOOL] Saving tip: Tip 1: Preparing for your first software engineering job is ...
=== RAW RESULT ===
Tip 1: Preparing for your first software engineering job is an important step in launching your career. Here’s a structured approach to help you get ready:

### 1. **Technical Skills Development**
   - **Programming Languages**: Ensure you are proficient in at least one or two programming languages commonly used in the industry (e.g., Python, Java, JavaScript, C#).
   - **Data Structures and Algorithms**: Understand fundamental data structures (arrays, lists, trees, graphs) and algorithms (sorting, searching). Resources like LeetCode, HackerRank, or GeeksforGeeks can be helpful.
   - **Version Control Systems**: Familiarize yourself with Git, including concepts like branching, merging, and pull requests.
   - **Development Tools**: Get comfortable with integrated development environments (IDEs) and text editors (e.g., VSCode, IntelliJ, Eclipse).

### 2. **Practical Experience**
   - **

### What you had to do manually above:

| Concern | Manual effort |
|---|---|
| Run an agent | Copy-paste `client.chat.completions.create()` for every agent |
| Tool use | Write full JSON schema by hand + dispatch `if msg.tool_calls` |
| Guardrail | A whole extra LLM call + fragile JSON parsing |
| Parallel execution | Would need manual `asyncio.gather()` wiring |
| Tracing / debugging | Nothing — blind |
| Handoff between agents | Pass strings manually and hope |

---
## ✅ APPROACH 2 — OpenAI Agents SDK (2_openai style)

Same pipeline. Watch how much disappears.

In [ ]:
from agents import (
    Agent, Runner, trace, function_tool,
    input_guardrail, GuardrailFunctionOutput,
    RunConfig
)
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from pydantic import BaseModel

# ── One-time client setup ────────────────────────────────────────
async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
model = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=async_client)
run_cfg = RunConfig(model=model)

In [ ]:
# ── ✅ Benefit 1: Tool = one decorator, zero JSON schema ─────────
@function_tool
def save_tip(tip: str) -> str:
    """Save a career tip to the user's notebook."""
    print(f"[TOOL] Saving tip: {tip[:60]}...")
    return "Saved successfully"
# That's it. The SDK auto-generates the JSON schema from the type hints + docstring.

In [ ]:
# ── ✅ Benefit 2: Guardrail = a decorator, not a separate pipeline ──
class CareerCheck(BaseModel):
    is_career_related: bool
    reason: str

guardrail_agent = Agent(
    name="Guardrail",
    instructions="Decide if the question is career-related. Be strict.",
    output_type=CareerCheck,  # Structured output — no manual JSON parsing
)

@input_guardrail
async def career_guardrail(ctx, agent, input):
    result = await Runner.run(guardrail_agent, input, run_config=run_cfg)
    check: CareerCheck = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_career_related
    )

In [ ]:
# ── ✅ Benefit 3: Agents are objects, not functions ───────────────
coach_professional = Agent(
    name="Professional Coach",
    instructions="You are a professional career coach. Give formal, structured career advice.",
    input_guardrails=[career_guardrail],
)

coach_motivational = Agent(
    name="Motivational Coach",
    instructions="You are an energetic motivational career coach. Give uplifting, passionate advice.",
    input_guardrails=[career_guardrail],
)

coach_concise = Agent(
    name="Concise Coach",
    instructions="You are a busy career coach. Give advice in bullet points, under 60 words.",
    input_guardrails=[career_guardrail],
)

picker = Agent(
    name="Picker",
    instructions=(
        "You are given 3 career tips. Pick the best one for a junior developer. "
        "Reply with ONLY the selected tip, no explanation."
    ),
    tools=[save_tip],  # ✅ Tool attached directly — no JSON schema written
)

In [ ]:
# ── ✅ Benefit 4: Parallel execution = one asyncio.gather, no wiring ──
async def sdk_pipeline(question: str) -> str:
    with trace("Career Coach Pipeline"):   # ✅ Full trace with ONE line
        try:
            # Parallel — all 3 coaches run simultaneously
            results = await asyncio.gather(
                Runner.run(coach_professional, question, run_config=run_cfg),
                Runner.run(coach_motivational, question, run_config=run_cfg),
                Runner.run(coach_concise,      question, run_config=run_cfg),
            )
        except Exception as e:
            # ✅ Guardrail tripwire surfaces here cleanly
            return f"Blocked: {e}"

        tips = [r.final_output for r in results]
        combined = "\n\n".join(f"Tip {i+1}:\n{t}" for i, t in enumerate(tips))

        # ✅ Picker also auto-calls save_tip tool — no manual dispatch
        final = await Runner.run(
            picker,
            f"{combined}\n\nAfter picking, please save the best tip.",
            run_config=run_cfg
        )
        return final.final_output

result_sdk = await sdk_pipeline("How should I prepare for my first software engineering job?")
print("=== SDK RESULT ===")
print(result_sdk)

In [ ]:
# ── ✅ Benefit 5: Guardrail actually blocks off-topic questions ───
result_blocked = await sdk_pipeline("What is the capital of France?")
print(result_blocked)

---
## 📊 Side-by-Side Summary

| Feature | ❌ Raw API (1_foundations) | ✅ Agents SDK (2_openai) |
|---|---|---|
| **Define an agent** | Copy-paste `client.chat.completions.create()` each time | `Agent(name, instructions, model)` |
| **Add a tool** | Write full JSON schema by hand + manual `if tool_calls` dispatch | `@function_tool` decorator — schema auto-generated |
| **Guardrail** | Separate LLM call + fragile JSON parsing + manual check | `@input_guardrail` decorator + tripwire |
| **Parallel agents** | Manual `asyncio` threading | `asyncio.gather(Runner.run(...), ...)` |
| **Structured output** | `json.loads()` + hope it works | `output_type=MyPydanticModel` |
| **Tracing / debugging** | Nothing | `with trace("name"):` → full trace at platform.openai.com/traces |
| **Agent handoff** | Pass strings manually | `handoff=[other_agent]` — agent routes itself |

> **Bottom line**: The SDK replaces all the boilerplate plumbing — JSON schemas, tool dispatch,
> guardrail wiring, async management — so you focus on *what* the agents do, not *how* they communicate.